# Plot

In [ ]:
library(readr)
library(purrr)
library(dplyr)
library(stringr)
library(fuzzyjoin)
library(ggplot2)
library(dplyr)
library(tidyverse)
library(Matrix)
library(reshape2)
library(RColorBrewer)
library(dplyr)
library(rstatix)
library(tibble)

options(tibble.width = Inf)

## Plot parameters

In [ ]:
out_dir = "//ceph.groups/mshahbazi.grp/rsakata/Figures/ECAD_OE/output"
analysis_summary_files = c(
"/ceph.groups/mshahbazi.grp/rsakata/Figures/ECAD_OE/EXP91_92_94_ECAD_OE_counts_BF.csv")

In [ ]:
# define const for visualization
FONT.SIZE <- 7
LABEL.FONT.SIZE <- 7
w <- 2 
h <- 2.5
LINE.W <- 0.5/2.141959

# Set geom defaults globally
update_geom_defaults("line",      list(linewidth = LINE.W))
update_geom_defaults("errorbar",  list(linewidth = LINE.W))
#update_geom_defaults("point",     list(size = LINE.W, stroke = LINE.W))

settheme <- theme_minimal() + 
  theme(
    text = element_text(family = "sans"), 
    panel.background = element_blank(),
    panel.grid.major = element_blank(), 
    panel.grid.minor = element_blank(),
    plot.background = element_blank(),
    axis.ticks = element_line(colour = "black", linewidth = LINE.W),
    axis.ticks.length = unit(0.1, "cm"), 
    axis.line = element_line(linewidth = LINE.W, colour = "black"),
    axis.title = element_text(size = FONT.SIZE),
    axis.text = element_text(colour = "black", size = FONT.SIZE),
    strip.text = element_text(size = FONT.SIZE), 
    strip.text.y.left = element_text(angle = 0, hjust = 1, size = FONT.SIZE),
    legend.position = "right",
    legend.title = element_text(size = FONT.SIZE), 
    legend.text = element_text(size = FONT.SIZE),
    legend.key.size = unit(0.3, "cm"),
    axis.text.x = element_text(colour = "black", angle = 0, size = LABEL.FONT.SIZE),
    title = element_text(size = FONT.SIZE) 
  )


In [ ]:
color_condition = c("Control" = "#7BBA56", 
               "Reversine" = "#87549B", 
               "Mosaic" = "#E8973E")

col_Annexin= "#4db2cb"

col_ZVAD = c("ZVAD+" = "#6a6969", 
               "ZVAD-" = "#bbbbbb")

col_ZVAD2 = c("ZVAD+" = "#022047", 
               "ZVAD-" = "#157AFF")

col_GFP = "#7BBA56"
col_RFP = "#cb377c"


col_structure= c("developed" = "#4674b8",  
        "small.cavity" ="#F09938" , 
        "failed" = "#bcbec0")

## 1. Extract summary files

In [ ]:
merged_df <- read_csv(analysis_summary_files)


In [ ]:
head(merged_df)

merged_df %>%
  count(timepoint, Population, condition_1, condition_2, condition_3)

In [ ]:
df_sample  = merged_df  %>% filter(treatment != "zvad")
#%>% filter(EXP != 91)

# Plot 

### A) Proportion of structures 

In [ ]:
df_percent <- df_sample %>%
  group_by(condition, treatment, sample_name) %>%
  summarise(
    developed = sum(developed, na.rm = TRUE),
    failed = sum(failed, na.rm = TRUE),
    total = sum(total, na.rm = TRUE),
    percentage = developed / total * 100,
    .groups = "drop"
  )

df_percent

In [ ]:
melt_data =  melt(df_percent, id = c("sample_name", "treatment", "condition"), measure.vars = c("developed",'failed'),variable.name = "structure", value.name='frequency' )

melt_data$condition = factor(melt_data$condition, levels =unique(melt_data$condition))
head(melt_data)

In [ ]:
df_percent_plot <- melt_data  %>%
  group_by(sample_name, treatment, condition) %>%
  mutate(
    total = sum(frequency),
    percentage = frequency / total * 100
  ) %>%
  ungroup()

df_percent_plot

In [ ]:
title <- "proportion_developed_contingency_bytreatement"

w <- 3
h <- 2
options(repr.plot.width = w, repr.plot.height = h)

sample_order <- c("failed", "developed")

df_percent_plot <- df_percent_plot %>%
  mutate(
    structure = factor(structure, levels = sample_order),
    condition = factor(condition, levels = c("control", "mosaic")),
    treatment = factor(
      treatment,
      levels = c("none", "dox", "zvad", "dox_zvad")
    )
  )

p <- ggplot(
  df_percent_plot,
  aes(
    x = condition,
    y = percentage,
    fill = structure
  )
) +
  geom_col(
    position = "stack",
    alpha = 0.6,
    width = 0.8
  ) +
  facet_wrap(
    ~ treatment,
    nrow = 1
  ) +
  labs(
    title = title,
    y = "Proportion of structures (%)",
    x = NULL,
    fill = NULL
  ) +
  settheme +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    legend.key.size = unit(0.3, "cm")
  ) +
  scale_y_continuous(
    limits = c(0, 110),
    expand = c(0, 0)
  )+
      scale_fill_manual(values=col_structure)

ggsave(
  file.path(out_dir, sprintf("A_%s.pdf", title)),
  plot = p,
  width = w,
  height = h
)

p

In [ ]:
title <- "proportion_developed_contingency_bycondition"

w <- 3
h <- 2
options(repr.plot.width = w, repr.plot.height = h)

sample_order <- c("failed", "developed")
df_percent_plot <- df_percent_plot %>%
  mutate(
    structure = factor(structure, levels = sample_order),
    condition = factor(condition, levels = c("control", "mosaic")),
    treatment = factor(
      treatment,
      levels = c("none", "dox", "zvad", "dox_zvad")
    )
  )

p <- ggplot(
  df_percent_plot,
  aes(
    x = treatment,
    y = percentage,
    fill = structure
  )
) +
  geom_col(
    position = "stack",
    alpha = 0.6,
    width = 0.8
  ) +
  facet_wrap(
    ~ condition,
    nrow = 1
  ) +
  labs(
    title = title,
    y = "Proportion of structures (%)",
    x = NULL,
    fill = NULL
  ) +
  settheme +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    legend.key.size = unit(0.3, "cm")
  ) +
  scale_y_continuous(
    limits = c(0, 110),
    expand = c(0, 0)
  )+
      scale_fill_manual(values=col_structure)

ggsave(
  file.path(out_dir, sprintf("A_%s.pdf", title)),
  plot = p,
  width = w,
  height = h
)

p

In [ ]:

fisher_results <- df_percent %>%
  group_by(treatment) %>%
  group_modify(~ {

    mosaic_row <- .x %>%
      filter(condition == "mosaic")

    control_row <- .x %>%
      filter(condition == "control")

    contingency_table <- matrix(
      c(
        mosaic_row$developed,
        mosaic_row$failed,
        control_row$developed,
        control_row$failed
      ),
      nrow = 2,
      byrow = TRUE,
      dimnames = list(
        condition = c("mosaic", "control"),
        outcome = c("developed", "failed")
      )
    )

    test <- fisher.test(contingency_table)

    tibble(
      odds_ratio = unname(test$estimate),
      conf_low = test$conf.int[1],
      conf_high = test$conf.int[2],
      p_value = test$p.value
    )
  }) %>%
  ungroup() %>%
  mutate(
    p_adjusted = p.adjust(p_value, method = "holm"),
    significance = case_when(
      p_adjusted < 0.0001 ~ "****",
      p_adjusted < 0.001  ~ "***",
      p_adjusted < 0.01   ~ "**",
      p_adjusted < 0.05   ~ "*",
      TRUE                ~ "ns"
    )
  )

fisher_results

# quasibinomial model

In [ ]:
df_exp <- df_sample %>%
  filter(
    total > 0,
    !is.na(developed),
    !is.na(failed)
  ) %>%
  group_by(EXP, condition, treatment) %>%
  summarise(
    developed = sum(developed),
    failed = sum(failed),
    .groups = "drop"
  ) %>%
  mutate(
    EXP = factor(EXP),
    condition = factor(condition, levels = c("control", "mosaic")),
    treatment = factor(
      treatment,
      levels = c("none", "dox", "zvad", "dox_zvad")
    )
  )

In [ ]:
library(emmeans)

fit <- glm(
  cbind(developed, failed) ~ EXP + condition * treatment,
  family = binomial,
  data = df_exp
)

emm <- emmeans(
  fit,
  ~ condition | treatment
)

condition_comparisons <- contrast(
  emm,
  method = list(
    "mosaic vs control" = c(-1, 1)
  )
) |>
  rbind(adjust = "holm")  

summary(
  condition_comparisons,
  type = "response",
  infer = c(TRUE, TRUE)
)